In [ ]:
import json
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Graduation_Project/JSON Files/hammurabi_templates_flat.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# فحص نوع البيانات
print("نوع البيانات:", type(data))

if isinstance(data, dict):
    print("المفاتيح الموجودة في الملف:")
    print(list(data.keys()))
    # طباعة أول عنصر لمعاينة شكله
    first_key = list(data.keys())[0]
    print(f"\nمعاينة لمحتوى المفتاح '{first_key}':")
    print(data[first_key][:2] if isinstance(data[first_key], list) else data[first_key])

elif isinstance(data, list):
    print(f"الملف عبارة عن قائمة (List) تحوي {len(data)} عنصر.")
    print("\nمعاينة أول عنصر:")
    print(data[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
نوع البيانات: <class 'dict'>
المفاتيح الموجودة في الملف:
['title', 'total_templates', 'templates']

معاينة لمحتوى المفتاح 'title':
نماذج عقود


In [ ]:
import json
import re
import pandas as pd

# مسار الملف على Google Drive
file_path = '/content/drive/MyDrive/Graduation_Project/JSON Files/hammurabi_templates_flat.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# استخراج قائمة النماذج حسب بنية الملف
if isinstance(data, list):
    templates = data
    total_templates = len(templates)
elif isinstance(data, dict):
    templates = data.get('templates', [])
    total_templates = data.get('total_templates', len(templates))
else:
    templates = []
    total_templates = 0

print("عدد التمبلتس المعلن/الكلي:", total_templates)
print("عدد التمبلتس الفعلي في القائمة:", len(templates))

rows = []

for tpl in templates:
    doc_id = tpl.get('DocID', '')
    doc_cat = tpl.get('DocCat', '')
    subject = tpl.get('Subject', '')
    index_val = tpl.get('Index', '') or ''
    body = tpl.get('Body', '') or ''
    raw_body = tpl.get('RawBody', '')
    placeholders = tpl.get('Placeholders', [])

    # تنظيف نص الـ Body من التاغات مثل [[1:text]]
    clean_body = re.sub(r'\[\[\d+:[^\]]+\]\]', ' ', body)
    clean_body = re.sub(r'\s+', ' ', clean_body).strip()

    # تجهيز نص البحث
    formatted_index = index_val.replace(';', ', ') if index_val else ''
    search_text = f"{doc_cat}. {subject}. {formatted_index}. {clean_body[:400]}"

    rows.append({
        'doc_id': doc_id,
        'category': doc_cat,
        'subject': subject,
        'index_keywords': index_val,
        'search_text': search_text,
        'body': body,
        'raw_body': raw_body,
        'placeholders': placeholders
    })

# تحويل البيانات إلى DataFrame
df = pd.DataFrame(rows)
print("\nأبعاد البيانات (الصفوف، الأعمدة):", df.shape)
df[['doc_id', 'category', 'subject']].head(10)

عدد التمبلتس المعلن/الكلي: 211
عدد التمبلتس الفعلي في القائمة: 211

أبعاد البيانات (الصفوف، الأعمدة): (211, 8)


,doc_id,category,subject
0,Reg-403287257976090#2,عقود استثمار,عقد مطبوعات
1,390277833194024#2,تنازل و هبات,إقرار تنازل عن صك
2,390277833199024#2,تنازل و هبات,عقد لهبة في مقابل سداد دين مضمون بحق عيني على ...
3,390277833195024#2,تنازل و هبات,عقد رسمي لهبة في مقابل سداد ديون الواهب
4,390277833198024#2,تنازل و هبات,عقد رسمي لهبة مقترنة بشرط احتفاظ الواهب بحق ال...
5,390277833197024#2,تنازل و هبات,عقد رسمي لهبة قطعية بدون عوض من والد لولده القاصر
6,390277833196024#2,تنازل و هبات,عقد رسمي لهبة قطعية بدون عوض
7,390277833185024#2,وصية و وديعة,تخالص عن وديعة
8,390277833186024#2,وصية و وديعة,عقد وديعة بأجر
9,390277833183024#2,وصية و وديعة,إقرار بتسليم وديعة بلا أجر


In [ ]:
# تأكيد ما في DocID مكرر (ممكن يسبب مشاكل عند ما نرجع النتيجة النهائية)
print("عدد DocID مكرر:", df['doc_id'].duplicated().sum())

# توزيع الفئات
print(df['category'].value_counts())


عدد DocID مكرر: 0
category
عقود العمل - عقود عمل متنوعة                                    17
عقود بيع متنوعة - بيع العقارات                                  14
وصية و وديعة                                                    12
شركات التضامن والتوصية البسيطة                                  11
حقوق الارتفاق و الانتفاع و الرقبة - حق الانتفاع و حق الرقبة     10
إجارة و عقود التأجير                                            10
القروض و الرهن و العارية - الرهن                                 8
تحكيم و حراسة - تحكيم                                            7
حوالات الحق و الدين                                              7
عقود بيع متنوعة - إنذارات و سندات و فسخ                          6
اجراءات تأسيس الشركات                                            6
تنازل و هبات                                                     6
عقود العمل - عمل الأجانب وعمل السوريين لدى الأجانب               6
عقود التأمين                                                     6
عقود متنوعة تتعلق بالعقارات - قسمة 

In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.0 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-large')
# بديل أخف وأسرع لو حابة تجربي أول: 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
import numpy as np
import faiss

# تحضير النصوص بصيغة e5 (passage للمحتوى المخزّن)
passages = ["passage: " + t for t in df['search_text'].tolist()]

embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True  # مهم عشان نستخدم cosine similarity
)

embeddings = np.array(embeddings).astype('float32')

# بناء فهرس FAISS (Inner Product = cosine بعد normalize)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("عدد المتجهات بالفهرس:", index.ntotal)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

عدد المتجهات بالفهرس: 211


In [ ]:
save_path = '/content/drive/MyDrive/Graduation_Project/JSON Files/'

faiss.write_index(index, save_path + 'contracts.index')
df.to_pickle(save_path + 'contracts_df.pkl')

In [ ]:
def search_contracts(user_query, top_k=5):
    query_embedding = model.encode(
        ["query: " + user_query],
        normalize_embeddings=True
    ).astype('float32')

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'category': row['category'],
            'subject': row['subject'],
            'score': float(score)
        })
    return results


def get_contract_by_subject(subject_name, candidates):
    for c in candidates:
        if c['subject'].strip() == subject_name.strip():
            # نرجع للـ df الأصلي عشان نجيب الـ body
            match = df[
                (df['subject'] == c['subject']) &
                (df['category'] == c['category'])
            ]
            if not match.empty:
                return match.iloc[0]
    return None


def print_full_contract(row):
    """طباعة العقد كامل بشكل مقروء"""
    print(f"\n📄 الفئة: {row['category']}")
    print(f"📌 العنوان: {row['subject']}")
    print(f"📝 عدد الحقول القابلة للتعبئة: {len(row['placeholders'])}")
    print("─" * 60)
    print(row['raw_body'])

def interactive_search():
    user_query = input("🔍 اشرحيلي شو نوع العقد الي بدك ياه: ").strip()
    if not user_query:
        print("⚠️ ما كتبتي شي.")
        return

    candidates = search_contracts(user_query, top_k=5)

    print(f"\nهاي أقرب العقود لطلبك:\n")
    for i, c in enumerate(candidates, 1):
        print(f"{i}. {c['subject']}  [{c['category']}]")

    print()
    chosen_name = input("✏️ اكتبي اسم العقد الي بدك ياه بالضبط متل ما ظهر فوق: ").strip()

    chosen_row = get_contract_by_subject(chosen_name, candidates)

    if chosen_row is None:
        print("\n❌ ما لقيت هيك عقد بالقائمة. تأكدي إنك نسختي الاسم بالضبط.")
        return

    print_full_contract(chosen_row)


# تشغيل السيناريو
interactive_search()

🔍 اشرحيلي شو نوع العقد الي بدك ياه: بدي اعمل عقد اتنازل فيه عن دين لصديقي

هاي أقرب العقود لطلبك:

1. عقد رسمي لهبة في مقابل سداد ديون الواهب  [تنازل و هبات]
2. عقد لهبة في مقابل سداد دين مضمون بحق عيني على العقار الموهوب  [تنازل و هبات]
3. عقد رسمي لهبة مقترنة بشرط احتفاظ الواهب بحق الانتفاع  [تنازل و هبات]
4. عقد رسمي لهبة قطعية بدون عوض  [تنازل و هبات]
5. عقد رسمي بتنزيل قسط من بدل الدين  [عقود التأمين]

✏️ اكتبي اسم العقد الي بدك ياه بالضبط متل ما ظهر فوق: عقد رسمي لهبة في مقابل سداد ديون الواهب

📄 الفئة: تنازل و هبات
📌 العنوان: عقد رسمي لهبة في مقابل سداد ديون الواهب
📝 عدد الحقول القابلة للتعبئة: 34
────────────────────────────────────────────────────────────
الفريق الأول: السيد ___ بن ___ تولد ___ يحمل البطاقة الشخصية رقم ___ تاريخ ___ الصادرة عن أمانة السجل المدني في ___ والمسجل في ___ والمقيم في ___ بالمسكن رقم ___، الواهب


الفريق الثاني: السيد ___ بن ___، موهوب له


1- صرح الواهب ___ بأنني أملك تمام العقار المسجل على اسمي في الصحيفة العقارية رقم ___ من المنطقة العقارية ___ و 

In [ ]:
# ============================================================
# سيناريو تفاعلي كامل: بحث → اقتراح LLM → اختيار بالاسم → عرض العقد
# ============================================================

import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ---------- 1) البحث الدلالي ----------
def search_contracts(user_query, top_k=5):
    query_embedding = model.encode(
        ["query: " + user_query],
        normalize_embeddings=True
    ).astype('float32')

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'category': row['category'],
            'subject': row['subject'],
            'score': float(score)
        })
    return results


# ---------- 2) تحميل موديل مفتوح المصدر (lazy load) ----------
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # بدّليها لـ Qwen2.5-3B-Instruct لو الذاكرة ضيقة

_llm_tokenizer = None
_llm_model = None

def _load_llm():
    global _llm_tokenizer, _llm_model
    if _llm_model is None:
        print(f"⏳ عم يتحمّل الموديل {MODEL_NAME} ...")
        _llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        _llm_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("✅ تم تحميل الموديل")
    return _llm_tokenizer, _llm_model


# ---------- 3) طبقة LLM: اقتراح الأنسب من القائمة ----------
def suggest_best_match(user_query, candidates):
    """
    ترجع اقتراح الموديل كـ dict: {"choice": رقم أو None, "reason": "..."}
    ما بترجع اسم العقد مباشرة — بس بتقترح، واليوزر هو الي بيأكد بكتابة الاسم
    """
    tokenizer, llm = _load_llm()

    candidates_text = "\n".join(
        [f"{i+1}. الفئة: {c['category']} | العنوان: {c['subject']}"
         for i, c in enumerate(candidates)]
    )

    system_prompt = (
        "أنت مساعد قانوني متخصص بتصنيف نماذج العقود السورية. "
        "مهمتك الوحيدة: اقتراح رقم النموذج الأنسب من قائمة مرشحين بناءً على طلب المستخدم. "
        "جوابك يجب أن يكون حصراً بصيغة JSON صحيحة بدون أي نص إضافي قبلها أو بعدها."
    )

    user_prompt = f"""طلب المستخدم (مكتوب بلغة بسيطة أو عامية):
"{user_query}"

قائمة المرشحين المتاحين:
{candidates_text}

المطلوب:
- اختر رقم المرشح (من 1 إلى {len(candidates)}) الأنسب فعلياً لطلب المستخدم من ناحية الغاية القانونية للعقد (نوع التصرف، الأطراف، طبيعة العلاقة).
- إذا كان أكثر من مرشح مناسب، اختر الأدق تطابقاً مع تفاصيل الطلب.
- إذا ما في أي مرشح مناسب فعلياً، ارجع choice: 0.

أجب بصيغة JSON فقط بهذا الشكل بالضبط:
{{"choice": <رقم>, "reason": "<سبب الاختيار بجملة وحدة قصيرة بالعربي>"}}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(llm.device)

    output_ids = llm.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )
    response_text = tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    return {"choice": None, "reason": None, "raw_output": response_text}


# ---------- 4) استرجاع العقد بالاسم من نفس القائمة المعروضة ----------
def get_contract_by_subject(subject_name, candidates):
    for c in candidates:
        if c['subject'].strip() == subject_name.strip():
            match = df[
                (df['subject'] == c['subject']) &
                (df['category'] == c['category'])
            ]
            if not match.empty:
                return match.iloc[0]
    return None


def print_full_contract(row):
    print(f"\n📄 الفئة: {row['category']}")
    print(f"📌 العنوان: {row['subject']}")
    print(f"📝 عدد الحقول القابلة للتعبئة: {len(row['placeholders'])}")
    print("─" * 60)
    print(row['raw_body'])
    print("─" * 60)


# ---------- 5) الحلقة التفاعلية الكاملة ----------
def interactive_search(use_llm_suggestion=True):
    user_query = input("🔍 اشرحيلي شو نوع العقد الي بدك ياه: ").strip()
    if not user_query:
        print("⚠️ ما كتبتي شي.")
        return

    candidates = search_contracts(user_query, top_k=5)

    print(f"\nهاي أقرب العقود لطلبك:\n")
    for i, c in enumerate(candidates, 1):
        print(f"{i}. {c['subject']}  [{c['category']}]")

    if use_llm_suggestion:
        print("\n🤖 عم يفكر بأنسب خيار...")
        suggestion = suggest_best_match(user_query, candidates)
        choice_num = suggestion.get('choice')
        if choice_num and 1 <= choice_num <= len(candidates):
            best = candidates[choice_num - 1]
            print(f"   💡 الاقتراح: خيار رقم {choice_num} — {best['subject']}")
            print(f"   السبب: {suggestion.get('reason', '')}")
        else:
            print("   ⚠️ ما قدر الموديل يحدد اقتراح واضح، اختاري بنفسك من القائمة.")

    print()
    chosen_name = input("✏️ اكتبي اسم العقد الي بدك ياه بالضبط متل ما ظهر فوق: ").strip()

    chosen_row = get_contract_by_subject(chosen_name, candidates)

    if chosen_row is None:
        print("\n❌ ما لقيت هيك عقد بالقائمة. تأكدي إنك نسختي الاسم بالضبط.")
        return

    print_full_contract(chosen_row)


# تشغيل السيناريو (مع اقتراح LLM)
interactive_search(use_llm_suggestion=True)


هاي أقرب العقود لطلبك:

1. عقد رسمي لهبة في مقابل سداد ديون الواهب  [تنازل و هبات]
2. عقد لهبة في مقابل سداد دين مضمون بحق عيني على العقار الموهوب  [تنازل و هبات]
3. عقد رسمي لهبة مقترنة بشرط احتفاظ الواهب بحق الانتفاع  [تنازل و هبات]
4. عقد رسمي لهبة قطعية بدون عوض من والد لولده القاصر  [تنازل و هبات]
5. عقد هبة رسمي في مقابل ترتيب مرتب للواهب مدى حياته مضمون برهن  [عقود متنوعة تتعلق بالعقارات - متنوعة]

🤖 عم يفكر بأنسب خيار...
⏳ عم يتحمّل الموديل Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ تم تحميل الموديل
   💡 الاقتراح: خيار رقم 1 — عقد رسمي لهبة في مقابل سداد ديون الواهب
   السبب: لأنه يتناسب مع طلب المستخدم حيث أنه عقد رسمي لهبة في مقابل سداد ديون الواهب

